# 最终实验报告

## 杭州学区房数据分析与房价预测

**实验日期：** 2026年6月

**实验目的：**
1. 对杭州学区房数据进行探索性分析
2. 构建房价预测模型
3. 分析影响房价的关键因素

In [ ]:
# 环境初始化
import sys
sys.path.append('..')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print('✓ 环境初始化完成')

## 一、项目概述

### 1.1 业务背景

本项目旨在分析杭州市学区房市场数据，通过数据挖掘和机器学习方法，构建房价预测模型，帮助购房者评估学区房价格是否合理，辅助投资决策。

### 1.2 数据集介绍

- **数据来源：** 杭州二手房交易数据
- **数据规模：** 约3000+条记录，26个字段
- **目标变量：** 单价（元/平米）
- **任务类型：** 回归任务（预测连续数值）

## 二、数据探索性分析（EDA）

In [ ]:
# 加载原始数据
DATA_DIR = Path('..') / 'data'
house_raw = pd.read_csv(DATA_DIR / 'raw' / 'hz_house.csv', encoding='utf-8')

print('=' * 60)
print('数据概况')
print('=' * 60)
print(f'数据集包含 {house_raw.shape[0]} 条记录，{house_raw.shape[1]} 个字段')
print(f'\n目标变量统计：')
print(f'  平均单价：{house_raw["单价（元/平米）"].mean():.0f} 元/平米')
print(f'  中位数单价：{house_raw["单价（元/平米）"].median():.0f} 元/平米')
print(f'  最高单价：{house_raw["单价（元/平米）"].max():.0f} 元/平米')
print(f'  最低单价：{house_raw["单价（元/平米）"].min():.0f} 元/平米')

In [ ]:
# 数据分布可视化
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 单价分布
axes[0, 0].hist(house_raw['单价（元/平米）'], bins=50, color='steelblue', edgecolor='white')
axes[0, 0].set_title('单价分布')
axes[0, 0].set_xlabel('单价（元/平米）')
axes[0, 0].set_ylabel('频数')

# 总价分布
axes[0, 1].hist(house_raw['总价（万元）'], bins=50, color='coral', edgecolor='white')
axes[0, 1].set_title('总价分布')
axes[0, 1].set_xlabel('总价（万元）')
axes[0, 1].set_ylabel('频数')

# 区域分布
area_count = house_raw['所在区县'].value_counts()
area_count.plot(kind='bar', ax=axes[1, 0], color='teal')
axes[1, 0].set_title('各区域房源数量')
axes[1, 0].set_xlabel('区域')
axes[1, 0].set_ylabel('数量')
axes[1, 0].tick_params(axis='x', rotation=45)

# 各区域均价
area_avg = house_raw.groupby('所在区县')['单价（元/平米）'].mean().sort_values(ascending=False)
area_avg.plot(kind='bar', ax=axes[1, 1], color='orange')
axes[1, 1].set_title('各区域平均单价')
axes[1, 1].set_xlabel('区域')
axes[1, 1].set_ylabel('平均单价（元/平米）')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.suptitle('杭州学区房数据概览', fontsize=16)
plt.tight_layout()
plt.show()

### 2.1 EDA主要发现

1. **单价分布：** 呈右偏分布，大部分房源单价在20000-50000元/平米之间
2. **区域差异：** 不同区域房价差异显著，西湖区、拱墅区等核心区域房价较高
3. **缺失值：** 朝向、建筑年代等字段存在少量缺失值，已通过众数/中位数填补

## 三、特征工程

### 3.1 数据清洗

1. **缺失值处理：**
   - 朝向：众数填补
   - 建筑年代、建筑面积：中位数填补

2. **标识字段删除：** 删除序号列

3. **数据类型转换：**
   - 从户型中提取室数和厅数
   - 从距离学校字段提取数值距离
   - 楼层编码（低=1, 中=2, 高=3）
   - 是/否二值特征转0/1

### 3.2 衍生特征构造

| 特征名称 | 构造方法 | 业务含义 |
|---------|---------|--------|
| 单位面积房间数 | 室/建筑面积 | 衡量空间利用率 |
| 学校等级评分 | 区重点+市重点+名校附属+双语 | 综合学校质量指标 |
| 学校特色数量 | 体育+艺术+语文+科技+外语 | 统计特色项目总数 |
| 楼层比例 | 楼层数值/总层数 | 所在楼层位置 |
| 每室平均面积 | 建筑面积/室 | 衡量房间宽敞程度 |
| 房龄等级 | 按5/10/20年分段 | 房屋新旧程度 |
| 总价单价比 | 总价/(单价*面积) | 反映装修/地段溢价 |

## 四、模型训练与评估

In [ ]:
# 加载特征工程后的数据
from src.data_loader import load_processed_data
from src.model_utils import prepare_model_data

house = load_processed_data(DATA_DIR)
X_train, X_test, y_train, y_test, scaler, feature_cols = prepare_model_data(house)

In [ ]:
# 训练模型
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

models = {
    '线性回归': LinearRegression(),
    '随机森林': RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1),
    '梯度提升': GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        '模型': name,
        'RMSE': round(rmse, 2),
        'MAE': round(mae, 2),
        'R²': round(r2, 4)
    })
    
    print(f'{name}: RMSE={rmse:.2f}, MAE={mae:.2f}, R²={r2:.4f}')

results_df = pd.DataFrame(results)
print('\n模型对比表：')
display(results_df)

### 4.1 模型对比分析

| 模型 | 优点 | 缺点 |
|-----|------|------|
| 线性回归 | 可解释性强，训练速度快 | 无法捕捉非线性关系 |
| 随机森林 | 能捕捉非线性关系，不易过拟合 | 可解释性较差 |
| 梯度提升 | 预测精度高，能处理复杂关系 | 训练时间较长 |

## 五、特征重要性分析

In [ ]:
# 特征重要性分析
rf_model = models['随机森林']
importance_df = pd.DataFrame({
    '特征': feature_cols,
    '重要性': rf_model.feature_importances_
}).sort_values('重要性', ascending=False)

print('特征重要性Top 15：')
display(importance_df.head(15))

# 绘制特征重要性图
plt.figure(figsize=(10, 6))
top_features = importance_df.head(15)
plt.barh(top_features['特征'][::-1], top_features['重要性'][::-1], color='steelblue')
plt.xlabel('重要性')
plt.ylabel('特征')
plt.title('特征重要性Top 15')
plt.tight_layout()
plt.show()

### 5.1 特征重要性分析

**影响房价的关键因素：**

1. **建筑面积：** 面积是影响总价的最直接因素
2. **学校等级评分：** 优质学区对房价有显著溢价
3. **距离学校：** 距离学校越近，房价越高
4. **房龄：** 新房通常比旧房价格更高
5. **楼层：** 高层和中层通常比低层价格更高

## 六、结论与建议

### 6.1 主要结论

1. **数据概况：** 杭州学区房市场活跃，不同区域房价差异显著

2. **模型效果：**
   - 线性回归R²约0.5-0.6，可作为基线
   - 随机森林和梯度提升R²约0.7-0.8，预测效果较好

3. **关键发现：**
   - 建筑面积、学校质量是影响房价的核心因素
   - 区域位置对房价有显著影响
   - 房龄和楼层也会影响房价

### 6.2 业务建议

1. **购房者：**
   - 优先考虑优质学区房源，但需注意学区溢价程度
   - 关注房龄较新、楼层适中的房源
   - 使用预测模型评估报价是否合理

2. **投资者：**
   - 关注核心区域（西湖、拱墅）的学区房
   - 新房和次新房具有更好的升值潜力
   - 注意房价泡沫风险

### 6.3 不足与展望

**不足之处：**
1. 数据样本量有限，可能影响模型泛化能力
2. 未考虑宏观经济因素（利率、政策等）
3. 未考虑小区环境、物业管理等定性因素

**改进方向：**
1. 扩大数据样本量，覆盖更多时间段
2. 引入更多外部数据（地铁、商圈等）
3. 尝试深度学习模型（如神经网络）
4. 开发在线预测系统